# Prediction, step by step

This notebook follows one prediction from the raw tables to the final scorecard. At every
step it shows which function is called, which arrays go in and come out, their shapes,
and what the numbers mean.

Each stage repeats the body of a library function line by line, prints every
intermediate object with `peek`, and then checks with an `assert` that the result equals
what the library itself returns. The walkthrough therefore cannot drift away from the
real code without failing.

| Stage | What happens | Library code |
| --- | --- | --- |
| A | raw tables to the features of the players at risk | `features.snapshot_features` |
| B | features to an `xgboost.DMatrix` | `SurvivalModel._matrix` |
| C | booster: margin, leaves, SHAP contributions | `predict_margin`, `shap_risk` |
| D | margin to a survival curve, for Cox and for AFT | `predict_survival` |
| E | calibration of the curve | `IsotonicSurvivalCalibrator.transform` |
| F | curve to the scorecard | `pipeline.score_players` |
| G | check against what is possible by the churn definition | - |

Nothing is trained and nothing is written to disk: the notebook only reads `data/` and
`artifacts/`, so run `python -m churn.generate_data` and `python -m churn.pipeline` first.

In [1]:
import warnings

# Optuna imports tqdm.auto, which warns when ipywidgets is missing; no progress bar is used.
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import json
import sys
from collections.abc import Callable
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xgboost as xgb
from IPython.display import display
from plotly.colors import qualitative
from scipy.special import expit, ndtr

from churn import config, features
from churn import evaluation as ev
from churn.models import MODELS, AFTModel, CoxModel, SurvivalModel
from churn.pipeline import score_players

SNAPSHOT = config.SCORING_SNAPSHOT  # any date inside the data works
N_DEBUG = 5  # players followed through the prediction

DATA_DIR = config.DATA_DIR
ARTIFACTS_DIR = config.ARTIFACTS_DIR
MODELS_DIR = ARTIFACTS_DIR / "models"
TIMES = config.TIME_GRID  # days 1..85 on which survival curves are evaluated
HORIZON = config.HORIZON_DAYS
FEATURES = list(config.FEATURES)

pd.set_option("display.precision", 4, "display.max_columns", 40, "display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

if not (ARTIFACTS_DIR / "metrics.json").exists() or not (DATA_DIR / "activity.parquet").exists():
    raise FileNotFoundError(
        "data/ or artifacts/ is missing; run `uv run python -m churn.generate_data` "
        "and `uv run python -m churn.pipeline` first"
    )

## Tools: `peek` and the call tracer

`peek` prints the type, shape, dtype, range and first values of an object, together with
what it means, and adds a row to a journal that is shown as one table at the end.

`trace_calls` runs a function under `sys.setprofile` and prints every call that lands in
the `churn` package, indented by depth, with the shapes of the arguments and of the
returned value.

In [3]:
JOURNAL: list[dict[str, str]] = []
PREVIEW_DAYS = (1, 7, 14, 30, 60, 85)  # days shown when a (players, days) matrix is previewed
stage = "setup"  # set at the top of every stage; `peek` reads it


def peek(name: str, value: object, meaning: str, on_grid: bool = False) -> None:
    """Print what ``value`` is and what it means, and record it in the journal.

    ``on_grid=True`` marks a (players, days) matrix: its preview shows the days in
    ``PREVIEW_DAYS`` instead of the first columns.
    """
    kind, shape, dtype, stats = type(value).__name__, "", "", ""
    if isinstance(value, pd.DataFrame):
        shape, dtype = str(value.shape), "mixed"
        preview = value.head(3).to_string()
    elif isinstance(value, pd.Series | pd.Index | np.ndarray):
        array = np.asarray(value)
        shape, dtype = str(array.shape), str(array.dtype)
        if array.dtype.kind == "b":
            stats = f"true={array.sum()} false={(~array).sum()}"
        elif array.dtype.kind in "iuf" and array.size:
            numbers = array.astype(float)
            low, high, missing = np.nanmin(numbers), np.nanmax(numbers), np.isnan(numbers).sum()
            stats = f"min={low:.6g} max={high:.6g} nan={missing}"
        if isinstance(value, pd.Series):
            preview = value.head(4).to_string()
        elif on_grid:
            corner = array[:5, np.searchsorted(TIMES, PREVIEW_DAYS)]
            preview = f"days {', '.join(map(str, PREVIEW_DAYS))} (first rows):\n"
            preview += np.array2string(corner, precision=4, suppress_small=True)
        else:
            corner = array[:5, :6] if array.ndim == 2 else array[:8]
            preview = np.array2string(corner, precision=4, suppress_small=True, max_line_width=100)
    else:
        preview = repr(value)
    header = " ".join(part for part in (kind, shape and f"shape={shape}", dtype, stats) if part)
    print(f"[{stage}] {name}: {header}")
    print(f"    meaning: {meaning}")
    print("    " + preview.replace("\n", "\n    "))
    row = {"stage": stage, "object": name, "type": kind, "shape": shape, "dtype": dtype}
    JOURNAL.append(row | {"meaning": meaning})


def summarise(value: object) -> str:
    """One-word description of an argument or a return value for the call trace."""
    if isinstance(value, np.ndarray | pd.DataFrame | pd.Series):
        return f"{type(value).__name__}{value.shape}"
    if isinstance(value, tuple):
        return "(" + ", ".join(summarise(item) for item in value) + ")"
    if isinstance(value, pd.Timestamp):
        return str(value.date())
    if isinstance(value, int | float | str | bool | None):
        return repr(value)
    return type(value).__name__


def trace_calls(function: Callable[..., Any], *args: Any) -> Any:
    """Run ``function(*args)`` and print the calls it makes inside the ``churn`` package."""
    package_dir = str(config.ROOT / "churn")
    lines: list[str] = []
    depth = 0

    def profiler(frame: Any, event: str, arg: Any) -> None:
        nonlocal depth
        code = frame.f_code
        if not code.co_filename.startswith(package_dir) or code.co_name.startswith("<"):
            return
        indent = "    " * depth
        if event == "call":
            names = code.co_varnames[: code.co_argcount + code.co_kwonlyargcount]
            local = frame.f_locals
            shown = ", ".join(
                f"{n}={summarise(local[n])}" for n in names if n not in ("self", "cls")
            )
            owner = f"{type(local['self']).__name__}." if "self" in local else ""
            lines.append(f"{indent}{owner}{code.co_name}({shown})")
            depth += 1
        elif event == "return":
            depth -= 1
            lines.append(f"{'    ' * depth}  -> {summarise(arg)}")

    previous = sys.getprofile()
    sys.setprofile(profiler)
    try:
        result = function(*args)
    finally:
        sys.setprofile(previous)
    print("\n".join(lines))
    return result

## Inputs: models, calibrators, data and the players to follow

- `production` is the model that scores real players: the best configuration refitted on
  train + validation. Which class it is (Cox or AFT) is recorded in its meta file.
- `calibrator` is the isotonic calibrator fitted for the best model on validation.
- Stage D needs one model of each kind, so the evaluation model of the other kind is
  loaded as well, and every kind has its own calibrator.

The followed players are chosen to cover the whole range of `recency_days`, the feature
the prediction depends on most.

In [4]:
def load_model(name: str) -> SurvivalModel:
    """Load a fitted model; its class is recorded in the meta file."""
    prefix = MODELS_DIR / name
    meta = json.loads(Path(f"{prefix}.meta.json").read_text())
    return MODELS[meta["model"]].load(prefix)


def load_calibrator(name: str) -> ev.IsotonicSurvivalCalibrator:
    """Load the calibrator fitted for the evaluation model ``name``."""
    data = json.loads((MODELS_DIR / f"calibrator_{name}.json").read_text())
    return ev.IsotonicSurvivalCalibrator.from_dict(data)


best = json.loads((ARTIFACTS_DIR / "metrics.json").read_text())["best_model"]
production = load_model("production")
calibrators = {name: load_calibrator(name) for name in MODELS}
calibrator = calibrators[best]
cox = production if isinstance(production, CoxModel) else load_model("cox")
aft = production if isinstance(production, AFTModel) else load_model("aft")
print(f"production model: {type(production).__name__}, {production.num_boost_round} trees")
print(f"calibrator: fitted for '{best}' on days {calibrator.times_[0]}..{calibrator.times_[-1]}")

players, activity = features.load_raw(DATA_DIR)
churn_dates = features.first_churn_dates(activity, config.DATA_END)
frame = features.snapshot_features(players, activity, churn_dates, SNAPSHOT)

# One player near each of N_DEBUG evenly spaced recency values (lowest id on ties).
by_id = frame.sort_values("player_id")
targets = np.linspace(by_id["recency_days"].min(), by_id["recency_days"].max(), N_DEBUG)
picked = [(by_id["recency_days"] - target).abs().idxmin() for target in targets]
debug = frame.loc[list(dict.fromkeys(picked))]
print(f"snapshot {SNAPSHOT.date()}: {len(frame):,} players at risk, following {len(debug)}")
display(debug[["player_id", "recency_days", "active_days_7d", "active_days_30d", "tenure_days"]])

production model: CoxModel, 375 trees
calibrator: fitted for 'cox' on days 1..85
snapshot 2026-07-01: 4,328 players at risk, following 5


,player_id,recency_days,active_days_7d,active_days_30d,tenure_days
0,2,1,6,26,207
21,121,8,0,9,512
260,1194,16,0,7,271
231,1064,23,0,2,269
87,438,30,0,1,67


## Call map

The big picture before the details: the two library calls that turn raw tables into a
scorecard, with everything they call inside the package. Arguments and results are shown
as `Type(shape)`.

In [5]:
print("1. Features of everybody at risk on the snapshot\n")
trace_calls(features.snapshot_features, players, activity, churn_dates, SNAPSHOT)
print("\n2. Scoring of the followed players\n")
_ = trace_calls(score_players, production, calibrator, debug)

1. Features of everybody at risk on the snapshot

snapshot_features(players=DataFrame(20000, 5), activity=DataFrame(1700671, 9), churn_dates=Series(15672,), snapshot=2026-07-01)
  -> DataFrame(4328, 18)

2. Scoring of the followed players

score_players(model=CoxModel, calibrator=IsotonicSurvivalCalibrator, scoring=DataFrame(5, 18))
    CoxModel.predict_survival(X=DataFrame(5, 15), times=ndarray(85,))
        CoxModel.predict_margin(X=DataFrame(5, 15))
            CoxModel._matrix(X=DataFrame(5, 15))
              -> DMatrix
          -> ndarray(5,)
        CoxModel.baseline_cumulative_hazard(times=ndarray(85,))
          -> ndarray(85,)
      -> ndarray(5, 85)
    IsotonicSurvivalCalibrator.transform(surv=ndarray(5, 85), times=ndarray(85,))
      -> ndarray(5, 85)
    CoxModel.shap_risk(X=DataFrame(5, 15))
        CoxModel._matrix(X=DataFrame(5, 15))
          -> DMatrix
      -> ndarray(5, 15)
    expected_days(surv=ndarray(5, 85), times=ndarray(85,))
        _require_daily_grid(time

## Stage A. Raw tables to features

`features.snapshot_features` decides who is at risk on the snapshot date and aggregates
their recent activity. The steps below are the ones that matter for the logic: only
activity strictly before the snapshot is visible, and a player is at risk if they have
played before and their first churn has not happened yet.

In [6]:
stage = "A"
peek("players", players, "one row per player: registration date and static traits")
peek("activity", activity, "one row per player and active day")
peek("churn_dates", churn_dates, "date of the first churn by player_id; absent = never churned")

history = activity[activity["date"] < SNAPSHOT]
peek("history", history, "activity strictly before the snapshot: all the model may see")
last_active = history.groupby("player_id")["date"].max()
peek("last_active", last_active, "last active day of every player who has played")
churn = churn_dates.reindex(last_active.index)
still_at_risk = (churn.isna() | (churn >= SNAPSHOT)).to_numpy()
peek("still_at_risk", still_at_risk, "True if the first churn has not happened before the snapshot")
at_risk = last_active.index[still_at_risk]
peek("at_risk", at_risk, "player ids that get a row and a prediction")
window = history[history["date"] >= SNAPSHOT - pd.Timedelta(days=config.INACTIVITY_DAYS)]
peek("window", window, "activity of the last 30 days, the source of the *_30d features")

# Three features recomputed by hand and compared with the library's output.
recency = (SNAPSHOT - last_active.reindex(at_risk)).dt.days
active_30d = window.groupby("player_id").size().reindex(at_risk)
turnover_30d = window.groupby("player_id")["turnover"].sum().reindex(at_risk)
peek("recency_days", recency, "days since the last activity, 1 = played yesterday")
peek("frame", frame, "output of snapshot_features: ids, snapshot date, 15 features, churn date")

assert frame["player_id"].tolist() == at_risk.tolist()
assert (frame["recency_days"].to_numpy() == recency.to_numpy()).all()
assert (frame["active_days_30d"].to_numpy() == active_30d.to_numpy()).all()
assert np.allclose(frame["turnover_30d"], turnover_30d)
print("\nOK: the at-risk set and the recomputed features equal features.snapshot_features")

[A] players: DataFrame shape=(20000, 5) mixed
    meaning: one row per player: registration date and static traits
       player_id registration_date  age acquisition_channel favorite_game
    0          1        2024-03-22   23             organic   crash_games
    1          2        2025-12-06   50           affiliate         slots
    2          3        2025-08-19   41           affiliate         slots
[A] activity: DataFrame shape=(1700671, 9) mixed
    meaning: one row per player and active day
       player_id       date  sessions  bets  turnover    ggr  deposits  withdrawals  bonus
    0       2316 2024-01-01         1    73     33.61  16.18     20.93          0.0    0.0
    1       2442 2024-01-01         1    70     90.95  28.54    109.15          0.0    0.0
    2       3674 2024-01-01         1    42     19.73   0.79     16.66          0.0    0.0
[A] churn_dates: Series shape=(15672,) datetime64[us]
    meaning: date of the first churn by player_id; absent = never churned
 

## Stage B. Features to a DMatrix

The model sees only the 15 feature columns. XGBoost reads the two categorical columns as
integer codes and handles them natively (`enable_categorical=True`), so the categories
must be declared in the same order as in training; `snapshot_features` takes that order
from `config.CATEGORICAL_FEATURES`.

In [7]:
stage = "B"
X = debug[FEATURES]
peek("X", X, "feature matrix of the followed players, one row per player")
display(X.dtypes.astype(str).rename("dtype").to_frame())

codes = X[list(config.CATEGORICAL_FEATURES)].apply(lambda column: column.cat.codes)
peek("category codes", codes, "the integers XGBoost sees instead of the category names")

matrix = xgb.DMatrix(X[production.feature_names_], enable_categorical=True)
peek("matrix.num_row()", matrix.num_row(), "players in the DMatrix")
peek("matrix.num_col()", matrix.num_col(), "features in the DMatrix")
peek("matrix.feature_types", matrix.feature_types, "type flag per feature, 'c' = categorical")
assert matrix.feature_names == production.feature_names_

[B] X: DataFrame shape=(5, 15) mixed
    meaning: feature matrix of the followed players, one row per player
         tenure_days  recency_days  active_days_7d  active_days_30d  activity_ratio_7d_30d  sessions_30d  turnover_30d  ggr_30d  deposits_30d  withdrawals_30d  bonus_30d  avg_stake_30d  age acquisition_channel favorite_game
    0            207             1               6               26                  0.989            53       4318.95   407.72       1103.87            25.97       6.86         1.3812   50           affiliate         slots
    21           512             8               0                9                  0.000            20        841.02    99.47        124.86             6.24      26.86         0.6871   24            paid_ads         slots
    260          271            16               0                7                  0.000            15       1983.46    67.84        362.85            30.42       0.00         4.5181   29           affiliate   table_g

,dtype
tenure_days,int64
recency_days,int64
active_days_7d,int64
active_days_30d,int64
activity_ratio_7d_30d,float64
sessions_30d,int64
turnover_30d,float64
ggr_30d,float64
deposits_30d,float64
withdrawals_30d,float64


[B] category codes: DataFrame shape=(5, 2) mixed
    meaning: the integers XGBoost sees instead of the category names
         acquisition_channel  favorite_game
    0                      1              0
    21                     2              0
    260                    1              2
[B] matrix.num_row(): int
    meaning: players in the DMatrix
    5
[B] matrix.num_col(): int
    meaning: features in the DMatrix
    15
[B] matrix.feature_types: list
    meaning: type flag per feature, 'c' = categorical
    ['int', 'int', 'int', 'int', 'float', 'int', 'float', 'float', 'float', 'float', 'float', 'float', 'int', 'c', 'c']


## Stage C. The booster

The booster is a sum of regression trees. Its raw output, the **margin**, is one number
per player:

- Cox: `eta(x)`, the log hazard ratio. Larger means earlier churn.
- AFT: `mu(x)`, the location of `log T`. Larger means later churn.

Three more views of the same trees: the margin after the first `k` trees, the leaf each
player falls into in every tree, and the TreeSHAP split of the margin into one
contribution per feature plus a bias.

In [8]:
stage = "C"
margin_meaning = {
    "cox": "eta(x): log hazard ratio, larger = earlier churn",
    "aft": "mu(x): location of log(days to churn), larger = later churn",
}[production.name]
booster = production.booster_
peek("booster.num_boosted_rounds()", booster.num_boosted_rounds(), "trees in the model")

eta = booster.predict(matrix, output_margin=True).astype(float)
peek("margin", eta, margin_meaning)
assert np.allclose(eta, production.predict_margin(X))

transformed = booster.predict(matrix)
peek("booster.predict(matrix)", transformed, "without output_margin XGBoost returns exp(margin)")
assert np.allclose(transformed, np.exp(eta), rtol=1e-4)

checkpoints = [k for k in (1, 5, 10, 25, 50, 100, 200, 300) if k < production.num_boost_round]
checkpoints.append(production.num_boost_round)
by_trees = np.stack(
    [booster.predict(matrix, output_margin=True, iteration_range=(0, k)) for k in checkpoints]
)
peek("margin by number of trees", by_trees, "rows = checkpoints, columns = players")
assert np.allclose(by_trees[-1], eta)

leaves = booster.predict(matrix, pred_leaf=True)
peek("leaves", leaves, "leaf index of every player in every tree: (players, trees)")

contributions = booster.predict(matrix, pred_contribs=True)
peek("contributions", contributions, "TreeSHAP: one column per feature, bias in the last one")
assert np.allclose(contributions.sum(axis=1), eta, rtol=1e-4, atol=1e-3)
shap = production.risk_sign * contributions[:, :-1].astype(float)
peek("shap (risk-oriented)", shap, "contributions signed so that positive = pushes towards churn")
assert np.allclose(shap, production.shap_risk(X))
print("\nOK: margin, exp(margin), the sum of SHAP contributions and shap_risk all agree")

[C] booster.num_boosted_rounds(): int
    meaning: trees in the model
    375
[C] margin: ndarray shape=(5,) float64 min=-2.40618 max=12.3131 nan=0
    meaning: eta(x): log hazard ratio, larger = earlier churn
    [-2.4062  0.8004  3.1602  6.6174 12.3131]
[C] booster.predict(matrix): ndarray shape=(5,) float32 min=0.0901592 max=222604 nan=0
    meaning: without output_margin XGBoost returns exp(margin)
    [     0.0902      2.2263     23.5754    748.0218 222603.73  ]
[C] margin by number of trees: ndarray shape=(9, 5) float32 min=-2.49517 max=12.3131 nan=0
    meaning: rows = checkpoints, columns = players
    [[-0.0308  0.0362  0.0362  0.0362  0.0362]
     [-0.1648  0.1701  0.1701  0.1701  0.1701]
     [-0.3323  0.3376  0.3376  0.3376  0.3376]
     [-0.8348  0.6275  0.8401  0.8401  0.8401]
     [-1.6444  0.7696  1.6384  1.6776  1.6776]]
[C] leaves: ndarray shape=(5, 375) float32 min=2 max=120 nan=0
    meaning: leaf index of every player in every tree: (players, trees)
    [[105.  54.

In [9]:
labels = [
    f"player {player_id} (recency {recency_days})"
    for player_id, recency_days in zip(debug["player_id"], debug["recency_days"], strict=True)
]
fig = go.Figure()
for column, label in enumerate(labels):
    color = qualitative.Plotly[column % len(qualitative.Plotly)]
    trace = go.Scatter(x=checkpoints, y=by_trees[:, column], name=label, line={"color": color})
    fig.add_trace(trace)
fig.update_layout(
    title="Stage C: how the margin builds up as trees are added",
    xaxis_title="number of trees used",
    yaxis_title=f"margin ({production.name})",
    template="plotly_white",
    height=380,
)
fig.show()

## Stage D. Margin to a survival curve

Both models turn one number per player into a whole curve `S(t | x)`, the probability of
not having churned by day `t`, on the grid of days 1..85. The result is a matrix with
one row per player and one column per day.

### D1. Cox

`S(t | x) = exp(-H0(t) * exp(eta(x)))`. `H0` is the Breslow baseline cumulative hazard
estimated on the training data and stored with the model as a step function: its jump
times and its value after each jump.

In [10]:
stage = "D1"
role = "production" if cox is production else "evaluation"
print(f"Cox model in use: the {role} model\n")

eta_cox = cox.predict_margin(X)
peek("eta", eta_cox, "log hazard ratio per player")
hazard_ratio = np.exp(eta_cox)
peek("hazard_ratio", hazard_ratio, "exp(eta): how many times the baseline hazard the player has")

baseline_times = np.asarray(cox.state_["baseline_times"])
baseline_hazard = np.asarray(cox.state_["baseline_hazard"])
peek("baseline_times", baseline_times, "days on which the training data had churn events")
peek("baseline_hazard", baseline_hazard, "H0 right after each of those days (non-decreasing)")

step = np.searchsorted(baseline_times, TIMES, side="right")
peek("step", step, "for each grid day: how many jumps of H0 have happened by then")
H0 = np.concatenate([[0.0], baseline_hazard])[step]
peek("H0", H0, "baseline cumulative hazard on the grid days 1..85")
cumulative_hazard = np.outer(hazard_ratio, H0)
peek(
    "cumulative_hazard",
    cumulative_hazard,
    "H(t | x) = exp(eta) * H0(t): (players, days)",
    on_grid=True,
)
surv_cox = np.exp(-cumulative_hazard)
peek("S (Cox, raw)", surv_cox, "survival probability, row = player, column = day", on_grid=True)

assert np.allclose(surv_cox, cox.predict_survival(X, TIMES))
print("\nOK: equals CoxModel.predict_survival")

Cox model in use: the production model

[D1] eta: ndarray shape=(5,) float64 min=-2.40618 max=12.3131 nan=0
    meaning: log hazard ratio per player
    [-2.4062  0.8004  3.1602  6.6174 12.3131]
[D1] hazard_ratio: ndarray shape=(5,) float64 min=0.0901592 max=222604 nan=0
    meaning: exp(eta): how many times the baseline hazard the player has
    [     0.0902      2.2263     23.5754    748.0218 222603.7354]
[D1] baseline_times: ndarray shape=(89,) float64 min=1 max=89 nan=0
    meaning: days on which the training data had churn events
    [1. 2. 3. 4. 5. 6. 7. 8.]
[D1] baseline_hazard: ndarray shape=(89,) float64 min=2.68562e-06 max=1.1245 nan=0
    meaning: H0 right after each of those days (non-decreasing)
    [0.     0.     0.     0.0001 0.0002 0.0005 0.0011 0.002 ]
[D1] step: ndarray shape=(85,) int64 min=1 max=85 nan=0
    meaning: for each grid day: how many jumps of H0 have happened by then
    [1 2 3 4 5 6 7 8]
[D1] H0: ndarray shape=(85,) float64 min=2.68562e-06 max=1.06925 na

### D2. AFT

`log T = mu(x) + sigma * Z`, so `S(t | x) = P(Z > (log t - mu(x)) / sigma)`. The noise
distribution of `Z` and its scale `sigma` are hyperparameters stored with the model.

In [11]:
stage = "D2"
role = "production" if aft is production else "evaluation"
print(f"AFT model in use: the {role} model, noise = {aft.distribution}, sigma = {aft.scale:.4f}\n")

mu = aft.predict_margin(X)
peek("mu", mu, "location of log(days to churn) per player")
peek("exp(mu)", np.exp(mu), "the same on the scale of days")
log_t = np.log(TIMES.astype(float))
peek("log_t", log_t, "log of the grid days")
z = (log_t[None, :] - mu[:, None]) / aft.scale
peek(
    "z",
    z,
    "standardised distance of each day from the player's location: (players, days)",
    on_grid=True,
)

# Survival function of the standardised noise Z, one per XGBoost AFT distribution.
noise_survival = {
    "normal": lambda value: ndtr(-value),
    "logistic": lambda value: expit(-value),
    "extreme": lambda value: np.exp(-np.exp(value)),
}[aft.distribution]
surv_aft = noise_survival(z)
peek("S (AFT, raw)", surv_aft, "survival probability, row = player, column = day", on_grid=True)

assert np.allclose(surv_aft, aft.predict_survival(X, TIMES))
print("\nOK: equals AFTModel.predict_survival")

AFT model in use: the evaluation model, noise = normal, sigma = 0.8176

[D2] mu: ndarray shape=(5,) float64 min=0.146608 max=5.64909 nan=0
    meaning: location of log(days to churn) per player
    [5.6491 3.5652 2.7538 1.9258 0.1466]
[D2] exp(mu): ndarray shape=(5,) float64 min=1.1579 max=284.033 nan=0
    meaning: the same on the scale of days
    [284.0334  35.3472  15.7024   6.8609   1.1579]
[D2] log_t: ndarray shape=(85,) float64 min=0 max=4.44265 nan=0
    meaning: log of the grid days
    [0.     0.6931 1.0986 1.3863 1.6094 1.7918 1.9459 2.0794]
[D2] z: ndarray shape=(5, 85) float64 min=-6.90895 max=5.25415 nan=0
    meaning: standardised distance of each day from the player's location: (players, days)
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[-6.909  -4.5291 -3.6813 -2.7492 -1.9015 -1.4755]
     [-4.3603 -1.9804 -1.1327 -0.2006  0.6471  1.0731]
     [-3.368  -0.9881 -0.1403  0.7918  1.6395  2.0655]
     [-2.3553  0.0245  0.8723  1.8044  2.6521  3.0781]
     [-0.1793  2.

## Stage E. Calibration

The raw curve of the production model is recalibrated day by day. For every day `t` the
calibrator holds a monotone map (the knots of an isotonic regression fitted on
validation) from the raw churn probability `1 - S(t | x)` to the observed frequency of
churn by day `t`. After the 85 maps are applied, a running maximum over the days makes
the churn probability non-decreasing in time again.

In [12]:
stage = "E"
surv_raw = production.predict_survival(X, TIMES)
churn_raw = 1 - surv_raw
peek("churn_raw", churn_raw, "raw P(churn by day t), (players, days)", on_grid=True)

k = int(np.flatnonzero(TIMES == HORIZON)[0])
knots_x, knots_y = calibrator.knots_[k]
peek(f"knots_x (day {HORIZON})", knots_x, "raw probabilities at which the map changes")
peek(f"knots_y (day {HORIZON})", knots_y, "calibrated probabilities at those points")
one_day = np.interp(churn_raw[:, k], knots_x, knots_y)
peek(f"calibrated churn (day {HORIZON})", one_day, "the day-60 map applied to the players")

mapped = np.column_stack(
    [np.interp(churn_raw[:, day], x, y) for day, (x, y) in enumerate(calibrator.knots_)]
)
peek("mapped", mapped, "all 85 maps applied column by column", on_grid=True)
monotone = np.maximum.accumulate(mapped, axis=1)
peek(
    "monotone",
    monotone,
    "running maximum over days: churn probability never decreases",
    on_grid=True,
)
raised = int((monotone > mapped).sum())
print(f"    cells raised by the running maximum: {raised} of {mapped.size}")
surv = 1 - monotone
peek("S (calibrated)", surv, "final survival curves used for scoring", on_grid=True)

assert np.allclose(surv, calibrator.transform(surv_raw, TIMES))
print("\nOK: equals IsotonicSurvivalCalibrator.transform")

[E] churn_raw: ndarray shape=(5, 85) float64 min=2.42133e-07 max=1 nan=0
    meaning: raw P(churn by day t), (players, days)
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[0.     0.0001 0.0024 0.0208 0.0614 0.0919]
     [0.     0.0025 0.0577 0.4042 0.7908 0.9075]
     [0.0001 0.0262 0.4673 0.9958 1.     1.    ]
     [0.002  0.5699 1.     1.     1.     1.    ]
     [0.45   1.     1.     1.     1.     1.    ]]
[E] knots_x (day 60): ndarray shape=(56,) float64 min=0.011704 max=1 nan=0
    meaning: raw probabilities at which the map changes
    [0.0117 0.0232 0.0236 0.0357 0.0359 0.053  0.0531 0.0599]
[E] knots_y (day 60): ndarray shape=(56,) float64 min=0 max=0.985591 nan=0
    meaning: calibrated probabilities at those points
    [0.     0.     0.0095 0.0095 0.0338 0.0338 0.0473 0.0473]
[E] calibrated churn (day 60): ndarray shape=(5,) float64 min=0.05 max=0.985591 nan=0
    meaning: the day-60 map applied to the players
    [0.05   0.8485 0.9856 0.9856 0.9856]
[E] mapped: ndarray sha

Both kinds of model go through exactly this code, each with its own calibrator. The
60-day churn probability of the followed players, before and after calibration:

In [13]:
at_horizon = {"player_id": debug["player_id"].to_numpy()}
for name, model in (("cox", cox), ("aft", aft)):
    raw_curves = model.predict_survival(X, TIMES)
    calibrated_curves = calibrators[name].transform(raw_curves, TIMES)
    peek(
        f"S ({name}, calibrated)", calibrated_curves, f"calibrated curves of '{name}'", on_grid=True
    )
    at_horizon[f"{name} raw"] = 1 - raw_curves[:, k]
    at_horizon[f"{name} calibrated"] = 1 - calibrated_curves[:, k]
display(pd.DataFrame(at_horizon).set_index("player_id"))

[E] S (cox, calibrated): ndarray shape=(5, 85) float64 min=0 max=1 nan=0
    meaning: calibrated curves of 'cox'
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     0.9959 0.95   0.9   ]
     [1.     1.     1.     0.3333 0.1515 0.1364]
     [1.     1.     0.4286 0.0559 0.0144 0.0115]
     [1.     0.1538 0.026  0.026  0.0144 0.0115]
     [0.     0.     0.     0.     0.     0.    ]]
[E] S (aft, calibrated): ndarray shape=(5, 85) float64 min=0 max=1 nan=0
    meaning: calibrated curves of 'aft'
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     1.     0.9654 0.9125]
     [1.     1.     1.     0.2794 0.1828 0.1579]
     [1.     1.     0.55   0.0619 0.0192 0.0192]
     [1.     0.1538 0.0408 0.0408 0.012  0.012 ]
     [0.     0.     0.     0.     0.     0.    ]]


,cox raw,cox calibrated,aft raw,aft calibrated
player_id,,,,
2,0.0614,0.0500,0.0286,0.0346
121,0.7908,0.8485,0.7412,0.8172
1194,1.0000,0.9856,0.9494,0.9808
1064,1.0000,0.9856,0.9960,0.9880
438,1.0000,1.0000,1.0000,1.0000


In [14]:
days = [0, *TIMES]
fig = go.Figure()
for row, label in enumerate(labels):
    color = qualitative.Plotly[row % len(qualitative.Plotly)]
    style = {"legendgroup": label, "line": {"color": color}}
    fig.add_trace(go.Scatter(x=days, y=[1.0, *surv[row]], name=label, **style))
    style = {"legendgroup": label, "line": {"color": color, "dash": "dot"}, "showlegend": False}
    fig.add_trace(go.Scatter(x=days, y=[1.0, *surv_raw[row]], name=label, **style))
fig.add_vline(x=HORIZON, line_dash="dash", line_color="#6b7280")
fig.update_layout(
    title="Stage E: survival curves of the followed players, raw (dotted) and calibrated (solid)",
    xaxis_title="days since the snapshot",
    yaxis_title="probability of not having churned",
    yaxis_tickformat=".0%",
    template="plotly_white",
    height=440,
)
fig.show()

## Stage F. Curve to scorecard

Everything a scorecard shows is read off the calibrated curve and the SHAP contributions:

- churn probability within `h` days: `1 - S(h | x)`;
- remaining useful life: `E[min(T, 85)] = 1 + S(1) + ... + S(84)` days;
- risk score: the margin, signed so that larger means riskier (it breaks ties between
  players with equal calibrated probabilities);
- risk segment: fixed thresholds on the 60-day probability;
- risk drivers: the three features with the largest positive SHAP contribution.

In [15]:
stage = "F"
churn_prob = {h: 1 - surv[:, TIMES == h][:, 0] for h in config.SCORE_HORIZONS}
for h, values in churn_prob.items():
    peek(f"churn_prob_{h}d", values, f"calibrated P(churn within {h} days)")
rul = 1.0 + surv[:, :-1].sum(axis=1)
peek("rul_days", rul, "expected days until churn, capped at 85")
risk_score = production.risk_sign * eta
peek("risk_score", risk_score, "margin signed so that larger = riskier")

names, bounds = zip(*config.RISK_SEGMENTS, strict=True)
segment = np.select([churn_prob[HORIZON] >= bound for bound in bounds], names, default=names[-1])
peek(
    "risk_segment",
    segment,
    f"thresholds on the {HORIZON}-day probability: {dict(config.RISK_SEGMENTS)}",
)

top = np.argsort(-shap, axis=1)[:, :3]
peek("top", top, "column indices of the three largest SHAP contributions per player")
drivers = np.array(FEATURES)[top]
peek("drivers", drivers, "the same as feature names")
driver_shap = np.take_along_axis(shap, top, axis=1)
peek("driver_shap", driver_shap, "their SHAP values on the risk-score scale")

[F] churn_prob_7d: ndarray shape=(5,) float64 min=0 max=1 nan=0
    meaning: calibrated P(churn within 7 days)
    [0.     0.     0.     0.8462 1.    ]
[F] churn_prob_14d: ndarray shape=(5,) float64 min=0 max=1 nan=0
    meaning: calibrated P(churn within 14 days)
    [0.     0.     0.5714 0.974  1.    ]
[F] churn_prob_30d: ndarray shape=(5,) float64 min=0.00413223 max=1 nan=0
    meaning: calibrated P(churn within 30 days)
    [0.0041 0.6667 0.9441 0.974  1.    ]
[F] churn_prob_60d: ndarray shape=(5,) float64 min=0.05 max=1 nan=0
    meaning: calibrated P(churn within 60 days)
    [0.05   0.8485 0.9856 0.9856 1.    ]
[F] rul_days: ndarray shape=(5,) float64 min=1 max=82.4592 nan=0
    meaning: expected days until churn, capped at 85
    [82.4592 34.4097 15.9292  7.6875  1.    ]
[F] risk_score: ndarray shape=(5,) float64 min=-2.40618 max=12.3131 nan=0
    meaning: margin signed so that larger = riskier
    [-2.4062  0.8004  3.1602  6.6174 12.3131]
[F] risk_segment: ndarray shape=(5,) <

The library does all of stages B-F in one call, `score_players`, and sorts the players
from the riskiest down. Its output must equal the hand-made numbers above. For the
scoring snapshot it must also equal what the pipeline saved in `artifacts/scores.parquet`
(there the rank is the place among all players at risk, here among the followed ones).

In [16]:
scores, curves = score_players(production, calibrator, debug)
library = scores.set_index("player_id").loc[debug["player_id"]]
for h, values in churn_prob.items():
    assert np.allclose(library[f"churn_prob_{h}d"], values)
assert np.allclose(library["rul_days"], rul)
assert np.allclose(library["risk_score"], risk_score)
assert (library["risk_segment"].to_numpy() == segment).all()
assert (library[["driver_1", "driver_2", "driver_3"]].to_numpy() == drivers).all()
print("OK: equals pipeline.score_players")

if SNAPSHOT == config.SCORING_SNAPSHOT:
    saved = pd.read_parquet(ARTIFACTS_DIR / "scores.parquet").set_index("player_id")
    saved = saved.loc[debug["player_id"]]
    for h, values in churn_prob.items():
        assert np.allclose(saved[f"churn_prob_{h}d"], values)
    assert np.allclose(saved["rul_days"], rul)
    print("OK: equals artifacts/scores.parquet for these players")

shown = ["risk_rank", "recency_days", *[f"churn_prob_{h}d" for h in config.SCORE_HORIZONS]]
shown += ["rul_days", "risk_score", "risk_segment", "driver_1", "driver_2", "driver_3"]
display(scores.set_index("player_id")[shown])

OK: equals pipeline.score_players


OK: equals artifacts/scores.parquet for these players


,risk_rank,recency_days,churn_prob_7d,churn_prob_14d,churn_prob_30d,churn_prob_60d,rul_days,risk_score,risk_segment,driver_1,driver_2,driver_3
player_id,,,,,,,,,,,,
438,1,30,1.0000,1.0000,1.0000,1.0000,1.0000,12.3131,high,recency_days,active_days_30d,active_days_7d
1064,2,23,0.8462,0.9740,0.9740,0.9856,7.6875,6.6174,high,recency_days,active_days_30d,favorite_game
1194,3,16,0.0000,0.5714,0.9441,0.9856,15.9292,3.1602,high,recency_days,active_days_7d,activity_ratio_7d_30d
121,4,8,0.0000,0.0000,0.6667,0.8485,34.4097,0.8004,high,active_days_7d,acquisition_channel,activity_ratio_7d_30d
2,5,1,0.0000,0.0000,0.0041,0.0500,82.4592,-2.4062,low,acquisition_channel,deposits_30d,bonus_30d


## Stage G. Check against the churn definition

Churn is 30 inactive days in a row, so a player last active `r` days before the snapshot
cannot churn before day `31 - r`. Neither model knows this: Cox gives every player the
same baseline time profile, AFT the same noise distribution, and the calibrator works
day by day. The check below measures how much churn probability a curve places on days
when churn is impossible for that player.

By day 60 every player at risk can have churned, so the 60-day probability is not
subject to this constraint; the short horizons are.

In [17]:
stage = "G"


def impossible_mass(curves: np.ndarray, earliest_day: np.ndarray) -> np.ndarray:
    """Predicted probability of churning before the earliest possible day."""
    last_impossible = earliest_day - 1  # 0 = churn is possible from day 1
    column = np.maximum(last_impossible, 1) - 1
    mass = 1 - curves[np.arange(len(curves)), column]
    return np.where(last_impossible >= 1, mass, 0.0)


earliest = (config.INACTIVITY_DAYS + 1 - X["recency_days"]).to_numpy()
peek("earliest", earliest, "first day on which each followed player can churn: 31 - recency")
check = pd.DataFrame(
    {
        "player_id": debug["player_id"].to_numpy(),
        "recency_days": X["recency_days"].to_numpy(),
        "earliest_possible_day": earliest,
        "raw P(churn before it)": impossible_mass(surv_raw, earliest),
        "calibrated P(churn before it)": impossible_mass(surv, earliest),
    }
)
display(check)

X_all = frame[FEATURES]
surv_all = calibrator.transform(production.predict_survival(X_all, TIMES), TIMES)
peek(
    "surv_all", surv_all, "calibrated curves of every player at risk on the snapshot", on_grid=True
)
earliest_all = (config.INACTIVITY_DAYS + 1 - X_all["recency_days"]).to_numpy()
mass_all = impossible_mass(surv_all, earliest_all)
summary = {
    "mean": mass_all.mean(),
    "median": np.median(mass_all),
    "95th percentile": np.percentile(mass_all, 95),
    "max": mass_all.max(),
    "share of players above 0.05": (mass_all > 0.05).mean(),
}
print(f"\nCalibrated probability placed on impossible days, {len(mass_all):,} players at risk:")
display(pd.Series(summary, name="value").to_frame())

[G] earliest: ndarray shape=(5,) int64 min=1 max=30 nan=0
    meaning: first day on which each followed player can churn: 31 - recency
    [30 23 15  8  1]


,player_id,recency_days,earliest_possible_day,raw P(churn before it),calibrated P(churn before it)
0,2,1,30,0.0193,0.0000
1,121,8,23,0.2254,0.4839
2,1194,16,15,0.4673,0.5714
3,1064,23,8,0.5699,0.8462
4,438,30,1,0.0000,0.0000


[G] surv_all: ndarray shape=(4328, 85) float64 min=0 max=1 nan=0
    meaning: calibrated curves of every player at risk on the snapshot
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     0.9959 0.95   0.9   ]
     [1.     1.     1.     0.9846 0.8443 0.7598]
     [1.     1.     1.     0.9926 0.9003 0.8329]
     [1.     1.     0.9697 0.1932 0.086  0.0538]
     [1.     1.     1.     0.9926 0.9003 0.8329]]

Calibrated probability placed on impossible days, 4,328 players at risk:


,value
mean,0.0564
median,0.0050
95th percentile,0.3725
max,0.9400
share of players above 0.05,0.1939


In [18]:
by_recency = pd.Series(mass_all).groupby(X_all["recency_days"].to_numpy()).agg(["mean", "size"])
peek("by_recency", by_recency, "mean probability on impossible days and players, by recency_days")
bars = go.Bar(x=by_recency.index, y=by_recency["mean"], customdata=by_recency["size"])
bars.hovertemplate = "recency %{x} days<br>mean %{y:.1%}<br>%{customdata} players<extra></extra>"
fig = go.Figure(bars)
fig.update_layout(
    title="Stage G: mean churn probability placed on impossible days, by recency",
    xaxis_title="recency_days (days since the last activity)",
    yaxis_title="mean probability",
    yaxis_tickformat=".0%",
    template="plotly_white",
    height=360,
)
fig.show()

[G] by_recency: DataFrame shape=(30, 2) mixed
    meaning: mean probability on impossible days and players, by recency_days
         mean  size
    1  0.0133  2120
    2  0.0174   772
    3  0.0378   444


## Journal

Every object shown above, in order: where it appears, its shape and what it is.

In [19]:
display(pd.DataFrame(JOURNAL))

,stage,object,type,shape,dtype,meaning
0,A,players,DataFrame,"(20000, 5)",mixed,one row per player: registration date and static traits
1,A,activity,DataFrame,"(1700671, 9)",mixed,one row per player and active day
2,A,churn_dates,Series,"(15672,)",datetime64[us],date of the first churn by player_id; absent = never churned
3,A,history,DataFrame,"(1700671, 9)",mixed,activity strictly before the snapshot: all the model may see
4,A,last_active,Series,"(20000,)",datetime64[us],last active day of every player who has played
5,A,still_at_risk,ndarray,"(20000,)",bool,True if the first churn has not happened before the snapshot
6,A,at_risk,Index,"(4328,)",int64,player ids that get a row and a prediction
7,A,window,DataFrame,"(89675, 9)",mixed,"activity of the last 30 days, the source of the *_30d features"
8,A,recency_days,Series,"(4328,)",int64,"days since the last activity, 1 = played yesterday"
9,A,frame,DataFrame,"(4328, 18)",mixed,"output of snapshot_features: ids, snapshot date, 15 features, churn date"
